In [ ]:
import pandas as pd

# ------------------------- STEP 1: LOAD DATA -------------------------

# Load patient demographics
patients = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/patients.csv")

# Load hospital admissions
admissions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/admissions.csv")

# Load ICU stays
icustays = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/icu/icustays.csv")

# Load diagnoses (ICD codes assigned to patients)
diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/diagnoses_icd.csv")

# Load prescriptions (medications administered)
prescriptions = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/prescriptions.csv")

# Load medical procedures (ICD codes for treatments)
procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/procedures_icd.csv")

# Load ICD descriptions (for both ICD-9 and ICD-10)
icd_diagnoses = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_diagnoses.csv.gz")
icd_procedures = pd.read_csv("/root/MIMICIV/data/mimic-iv-3.1/hosp/d_icd_procedures.csv.gz")

# Convert ICU stay timestamps to datetime format
icustays['intime'] = pd.to_datetime(icustays['intime'], errors='coerce')
icustays['outtime'] = pd.to_datetime(icustays['outtime'], errors='coerce')

# ------------------------- STEP 2: MAP ICD CODES TO TEXT DESCRIPTIONS -------------------------

# Merge diagnoses with descriptions
diagnoses = diagnoses.merge(icd_diagnoses, on=["icd_code", "icd_version"], how="left").rename(columns={'icd_version_x': 'icd_version'})
diagnoses['diagnosis_description'] = diagnoses['long_title'].fillna("Unknown diagnosis")

# Merge procedures with descriptions
procedures = procedures.merge(icd_procedures, on=["icd_code", "icd_version"], how="left").rename(columns={'icd_version_x': 'icd_version'})
procedures['procedure_description'] = procedures['long_title'].fillna("Unknown procedure")

In [ ]:
admissions[admissions['subject_id']== 11530780]

In [ ]:
# ------------------------- STEP 3: COMPUTE AGE & MERGE DATA -------------------------

# Keep only relevant patient information
patients = patients[['subject_id', 'anchor_age', 'anchor_year', 'gender']]

# Merge patient age & demographics into event tables
admissions = admissions.merge(patients, on='subject_id', how='left', suffixes=None)
icustays = icustays.merge(patients, on='subject_id', how='left', suffixes=None)
diagnoses = diagnoses.merge(patients, on='subject_id', how='left', suffixes=None)
procedures = procedures.merge(patients, on='subject_id', how='left', suffixes=None)
prescriptions = prescriptions.merge(patients, on='subject_id', how='left', suffixes=None)

# Convert date columns to datetime
admissions['admittime'] = pd.to_datetime(admissions['admittime'])
icustays['intime'] = pd.to_datetime(icustays['intime'])
prescriptions['starttime'] = pd.to_datetime(prescriptions['starttime'])

# Compute patient age at each event
admissions['age_at_event'] = admissions['anchor_age'] + (admissions['admittime'].dt.year - admissions['anchor_year'])
icustays['age_at_event'] = icustays['anchor_age'] + (icustays['intime'].dt.year - icustays['anchor_year'])
diagnoses['age_at_event'] = diagnoses['anchor_age']
procedures['age_at_event'] = procedures['anchor_age']
prescriptions['age_at_event'] = prescriptions['anchor_age']

In [ ]:
# ------------------------- STEP 4: BUILD A TABULAR DATASET -------------------------

# Aggregate data per hospitalization (`hadm_id`)
features = admissions[['subject_id', 'hadm_id', 'age_at_event', 'gender', 'admission_type', 'discharge_location']]

# Here we are referring to days
# ICU stays: Number of ICU admissions per hospitalization
# icu_summary = icustays.groupby('hadm_id').agg(
#     icu_admissions=('stay_id', 'count'),
#     icu_hours=('intime', lambda x: (x.max() - x.min()).total_seconds() / 3600)
# ).reset_index()

# Here to hours
icu_summary = icustays.assign(
    icu_duration_hours=(icustays['outtime'] - icustays['intime']).dt.total_seconds() // 3600
).groupby('hadm_id').agg(
    icu_admissions=('stay_id', 'count'),
    total_icu_hours=('icu_duration_hours', 'sum')
).reset_index()

# Diagnoses: Count number of diagnoses per hospitalization
diagnosis_summary = diagnoses.groupby('hadm_id').agg(
    num_diagnoses=('icd_code', 'count'),
    unique_diagnoses=('diagnosis_description', lambda x: list(x.unique()))
).reset_index()

# Procedures: Count number of procedures per hospitalization
procedure_summary = procedures.groupby('hadm_id').agg(
    num_procedures=('icd_code', 'count'),
    unique_procedures=('procedure_description', lambda x: list(x.unique()))
).reset_index()

# Medications: Count number of prescribed drugs per hospitalization
med_summary = prescriptions.groupby('hadm_id').agg(
    num_medications=('drug', 'count'),
    unique_medications=('drug', lambda x: list(x.unique()))
).reset_index()

# Merge all features into a single dataset
dataset = features.merge(icu_summary, on='hadm_id', how='left', suffixes=None)
dataset = dataset.merge(diagnosis_summary, on='hadm_id', how='left', suffixes=None)
dataset = dataset.merge(procedure_summary, on='hadm_id', how='left', suffixes=None)
dataset = dataset.merge(med_summary, on='hadm_id', how='left', suffixes=None)

# Fill missing values with 0 or "None" where appropriate
# To discuss whether is it properly appropriate or not.
dataset.fillna({'icu_admissions': 0, 'icu_hours': 0, 'num_diagnoses': 0, 'num_procedures': 0, 'num_medications': 0}, inplace=True)
dataset.fillna({'unique_diagnoses': 'None', 'unique_procedures': 'None', 'unique_medications': 'None'}, inplace=True)

In [ ]:
dataset.head(5)

In [ ]:
dataset.columns

In [ ]:
dataset.loc[2, 'unique_diagnoses']

In [ ]:
# ------------------------- STEP 5: SAVE & DISPLAY DATASET -------------------------

# Save the dataset to a CSV file for ML usage
dataset.to_csv("mimiciv_clinical_dataset.csv", index=False)


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load the dataset generated previously
dataset = pd.read_csv("mimiciv_clinical_dataset_tabular_death_visit.csv")

# ------------------------------------------
# 1. Distribution of number of visits per patient
# ------------------------------------------

# Count number of visits per patient
visits_per_patient = dataset.groupby('subject_id')['hadm_id'].nunique()

# Max number of visits for a single patient
max_visits = visits_per_patient.max()
print(f"Maximum number of visits for a single patient: {max_visits}")

# Plot histogram
plt.figure(figsize=(10,6))
plt.hist(visits_per_patient, bins=range(1, max_visits + 2), edgecolor='black', alpha=0.7)
plt.title("Distribution of Number of Visits per Patient")
plt.xlabel("Number of Visits")
plt.ylabel("Number of Patients")
plt.xticks(range(1, max_visits + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

# ------------------------------------------
# 2. Mortality Analysis and Ratios
# ------------------------------------------

# Total number of unique patients who died
patient_mortality = dataset.groupby('subject_id')['death_flag'].max().sum()

# Total deaths occurring specifically during visits
deaths_during_visit = dataset['died_during_visit'].sum()

# Calculate ratios
total_patients = dataset['subject_id'].nunique()
death_ratio = (patient_mortality / total_patients) * 100
in_hospital_death_ratio = (deaths_during_visit / patient_mortality * 100) if patient_mortality > 0 else 0

# Display the mortality results clearly
print(f"Total number of patients: {dataset['subject_id'].nunique()}")
print(f"Total number of patients who died: {int(patient_mortality)} ({patient_mortality/dataset['subject_id'].nunique():.2%})")
print(f"Deaths during hospitalization: {int(deaths_during_visit)} ({deaths_during_visit/patient_mortality:.2%} of all deaths)")


In [ ]:
visits_per_patient.max()

In [ ]:
# Identify patient(s) with the maximum number of visits
max_visits_patient = visits_per_patient[visits_per_patient == visits_per_patient.max()]
print("Patient(s) with the maximum number of visits:")
print(max_patient := max(visits_per_patient))
visits_per_patient[visits_per_patient == visits_per_patient.max()]

In [ ]:
# Select only rows for patient 15496609
patient_data = dataset[dataset['subject_id'] == '15496609']



In [ ]:
dataset.head()

In [ ]:
dataset[dataset['subject_id']==15496609]

Let's exclude this patient from our dataset

In [ ]:
# ------------------------------------------
# 1. Distribution of number of visits per patient
# ------------------------------------------

# Count number of visits per patient
visits_per_patient = dataset.groupby('subject_id')['hadm_id'].nunique()

# Max number of visits for a single patient
max_visits = visits_per_patient.max()
print(f"Maximum number of visits for a single patient: {max_visits}")

# Plot histogram
x_range = 20
plt.figure(figsize=(10,6))
plt.hist(visits_per_patient, bins=range(1, x_range + 2), edgecolor='black', alpha=0.7)
plt.title("Distribution of Number of Visits per Patient")
plt.xlabel("Number of Visits")
plt.ylabel("Number of Patients")
plt.xticks(range(1, x_range + 1))
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.show()

## Analysis

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Load dataset
dataset = pd.read_csv("mimiciv_clinical_dataset_tabular_death_visit.csv")

# Count admission categories
admission_counts = dataset['admission_category'].value_counts()

# Display counts
print("Admission category counts:")
print(admission_counts := dataset['admission_category'].value_counts())


In [ ]:
len(dataset) # 291667 + 254361

In [ ]:
dataset.head(5)